# SmolVLM-256M: Hybrid Attention (sdpa Prefill, eager Decode)

This notebook tests a mix of two attention implementations. The mix uses `sdpa` for prefill and `eager` for decode.

Earlier tests showed that `sdpa` is faster in prefill and uses less VRAM. They also showed that `eager` is faster in decode.

The script captures the decode graph with `eager` attention. Then it switches the model back to `sdpa`, and the prefill uses `sdpa`. A graph replay does not read the model setting, so the decode keeps the `eager` kernels.

The script runs three options on the same image and prompt: `sdpa`, `eager` and the mix (`hybrid`). It prints the prefill time, the decode time per token, the total time for prefill and decode, the peak VRAM, and whether the answer is the same as the answer of the first option.

In [16]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/hybrid_attn.py --image {REPO}/frame.jpg --max-len 1024

option   prefill/decode prefill ms decode ms/tok  total ms peak VRAM MB  same answer
Loading weights: 100% 471/471 [00:00<00:00, 1086.68it/s]
sdpa     sdpa/sdpa           257.7          8.14     778.7          873  True
Loading weights: 100% 471/471 [00:00<00:00, 1143.74it/s]
eager    eager/eager         383.0          6.43     794.4         2012  True
Loading weights: 100% 471/471 [00:00<00:00, 1559.51it/s]
hybrid   sdpa/eager          270.6          6.44     683.0          889  True


## Result

The test ran on a Colab GPU, with 875 prompt tokens, max_len 1024 and 64 new tokens. All options give the same answer.

| Option | Prefill / decode | Prefill (ms) | Decode (ms/token) | Total (ms) | Peak VRAM (MB) |
|---|---|---|---|---|---|
| sdpa | sdpa / sdpa | 274.7 | 8.14 | 795.9 | 873 |
| eager | eager / eager | 395.5 | 6.45 | 808.3 | 2012 |
| hybrid | sdpa / eager | 285.8 | 6.47 | 700.1 | 889 |

- The `hybrid` row shows `sdpa/eager`. The switch worked.
- Decode takes 6.47 ms per token, the same as `eager`. Prefill takes 286 ms, close to `sdpa`. The 11 ms difference is small, and we do not know the cause.
- The total for 64 tokens is 700 ms. This is 96 ms (12%) less than `sdpa` and 108 ms (13%) less than `eager`. The estimate was 107 ms. For the whole request with preprocessing, the gain is about 8%.
- Peak VRAM is 889 MB. This is 16 MB more than `sdpa` and less than half of `eager`.
- The gain grows with the answer length. `hybrid` is faster than `sdpa` for answers longer than about 7 tokens. For a 10-token answer, the gain is only about 6 ms.